[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PatrickJReed/soccer-vision/blob/master/examples/colab_homography_probe_v2.ipynb)

# Homography propagation probe v2 — drift control

v1 verdict: registration never fails (0/44), but naive **chained** propagation
drifts (median ~0.076 pitch-units; short gaps tight, long gaps drift). v2 tests
the two standard fixes against held-out ground truth:

1. **Direct-to-anchor** — register a gap frame *directly* to an anchor (one
   match), not by chaining every intermediate frame → no accumulated drift.
2. **Bidirectional blend** — propagate from the anchor *before* (A) and *after*
   (C) a held-out frame and blend by distance → pins both ends (loop closure).

**Held-out design:** take an anchor triple A → B → C with real gaps on both
sides. Propagate to **B** directly from A and from C; B's own landmarks are
*never used* in the propagation, so they are honest ground truth. Compare:
forward-only, backward-only, and blended error (pitch-units).

**Read:** if blended median drops well under v1's 0.076 (toward ~0.04) and holds
across gap sizes, drift control works and propagation is the fix. CPU-only.

In [ ]:
# Install (skip if running in the same session as a prior probe).
!rm -rf /content/soccer-vision
!git clone -q https://github.com/PatrickJReed/soccer-vision.git /content/soccer-vision
!pip install -q "/content/soccer-vision/packages/soccer-vision[roboflow]"

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from soccer_vision.pitch.landmarks import PITCH_LANDMARKS, build_frame_homographies

# Point this at wherever the Stage-1 checkpoint lives (Drive copy survives runtime restarts).
OUT = Path("/content/out")
if not (OUT / "keypoints.parquet").exists():
    OUT = Path("/content/drive/MyDrive/soccer-vision/out")
CLIP = "/content/drive/MyDrive/soccer-vision/bakeoff_clip.mp4"

kp = pd.read_parquet(OUT / "keypoints.parquet")
traj = pd.read_parquet(OUT / "trajectories_px.parquet")
H_pitch = build_frame_homographies(kp, conf_threshold=0.5)
anchors = sorted(H_pitch)
print(f"checkpoint: {OUT}")
print(f"{len(anchors)} anchor frames ({len(anchors) / 3733:.1%} of clip)")

In [ ]:
orb = cv2.ORB_create(3000)
bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)


def _mask(frame_idx, shape):
    m = np.full(shape[:2], 255, np.uint8)
    sel = traj[(traj.frame == frame_idx) & traj["class"].isin(["player", "goalkeeper", "referee"])]
    for _, r in sel.iterrows():
        cv2.rectangle(m, (int(r.bbox_x1) - 12, int(r.bbox_y1) - 12),
                      (int(r.bbox_x2) + 12, int(r.bbox_y2) + 12), 0, -1)
    return m


def register(f_src, img_src, f_dst, img_dst):
    """Homography mapping img_src pixels -> img_dst pixels (static background)."""
    ks, ds = orb.detectAndCompute(cv2.cvtColor(img_src, cv2.COLOR_BGR2GRAY), _mask(f_src, img_src.shape))
    kd, dd = orb.detectAndCompute(cv2.cvtColor(img_dst, cv2.COLOR_BGR2GRAY), _mask(f_dst, img_dst.shape))
    if ds is None or dd is None or len(ds) < 12 or len(dd) < 12:
        return None
    matches = bf.match(ds, dd)
    if len(matches) < 12:
        return None
    src = np.float32([ks[m.queryIdx].pt for m in matches])
    dst = np.float32([kd[m.trainIdx].pt for m in matches])
    G, _ = cv2.findHomography(src, dst, cv2.RANSAC, 3.0)
    return G


def read_frame(cap, idx):
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ok, f = cap.read()
    return f if ok else None


def map_px(H, px):
    pts = np.column_stack([px[:, 0], px[:, 1], np.ones(len(px))])
    m = (H @ pts.T).T
    m /= m[:, 2:3]
    return m[:, :2]


def landmark_targets(frame):
    kb = kp[(kp.frame == frame) & (kp.conf >= 0.5) & (kp.kp_idx < len(PITCH_LANDMARKS))]
    if len(kb) < 4:
        return None, None
    return kb[["x_px", "y_px"]].to_numpy(), PITCH_LANDMARKS[kb.kp_idx.to_numpy()]


def direct_H(cap, B, S, H_S):
    """Pitch homography for frame B propagated DIRECTLY from anchor S (no chaining)."""
    fb, fs = read_frame(cap, B), read_frame(cap, S)
    if fb is None or fs is None:
        return None
    W = register(B, fb, S, fs)            # B pixels -> S pixels
    return None if W is None else H_S @ W

In [ ]:
cap = cv2.VideoCapture(CLIP)
rows = []
for i in range(1, len(anchors) - 1):
    A, B, C = anchors[i - 1], anchors[i], anchors[i + 1]
    gapA, gapC = B - A, C - B
    if min(gapA, gapC) < 3 or max(gapA, gapC) > 120:   # real gap both sides, bounded
        continue
    px, target = landmark_targets(B)
    if px is None:
        continue
    H_f = direct_H(cap, B, A, H_pitch[A])
    H_b = direct_H(cap, B, C, H_pitch[C])
    e_f = float(np.linalg.norm(map_px(H_f, px) - target, axis=1).mean()) if H_f is not None else None
    e_b = float(np.linalg.norm(map_px(H_b, px) - target, axis=1).mean()) if H_b is not None else None
    w_f = gapC / (gapA + gapC)            # weight the nearer anchor more
    if H_f is not None and H_b is not None:
        blended = w_f * map_px(H_f, px) + (1 - w_f) * map_px(H_b, px)
    elif H_f is not None:
        blended = map_px(H_f, px)
    elif H_b is not None:
        blended = map_px(H_b, px)
    else:
        blended = None
    e_blend = float(np.linalg.norm(blended - target, axis=1).mean()) if blended is not None else None
    rows.append((A, B, C, gapA, gapC, e_f, e_b, e_blend))
    if len(rows) >= 60:
        break
cap.release()

print(f"{'A':>5} {'B':>5} {'C':>5} {'gA':>4} {'gC':>4}  {'fwd':>7} {'bwd':>7} {'blend':>7}")
for A, B, C, gA, gC, ef, eb, eblend in rows:
    f = lambda e: "  fail " if e is None else f"{e:.3f}"
    print(f"{A:5d} {B:5d} {C:5d} {gA:4d} {gC:4d}  {f(ef):>7} {f(eb):>7} {f(eblend):>7}")


def summary(name, vals):
    v = sorted(x for x in vals if x is not None)
    if not v:
        print(f"  {name:6}: no successes")
        return
    med = v[len(v) // 2]
    print(f"  {name:6}: median {med:.3f}   <=0.05 {sum(x <= 0.05 for x in v) / len(v):.0%}"
          f"   <=0.10 {sum(x <= 0.10 for x in v) / len(v):.0%}   (n={len(v)})")


print("\n--- verdict (v1 chained baseline median was 0.076) ---")
summary("fwd", [r[5] for r in rows])
summary("bwd", [r[6] for r in rows])
summary("blend", [r[7] for r in rows])

print("\nblend error by gap size (max of the two side-gaps):")
for lo, hi in [(3, 20), (21, 60), (61, 120)]:
    band = [r[7] for r in rows if lo <= max(r[3], r[4]) <= hi and r[7] is not None]
    if band:
        b = sorted(band)
        print(f"  gap {lo:3d}-{hi:3d}: median {b[len(b) // 2]:.3f}  (n={len(b)})")

## Visual — a held-out frame, blended-propagated

Scatter a held-out anchor B's players on the canonical pitch using the *blended*
propagated homography (B's own landmarks unused), with B's landmark reprojection
error in the title. Players should land in `[0,1]²` in a plausible shape.

In [ ]:
import matplotlib.pyplot as plt

cand = [r for r in rows if r[7] is not None]
if not cand:
    print("No blended success to visualize; see the table.")
else:
    cand.sort(key=lambda r: r[7])
    A, B, C, gA, gC, ef, eb, eblend = cand[len(cand) // 2]   # a median-quality case
    cap = cv2.VideoCapture(CLIP)
    H_f = direct_H(cap, B, A, H_pitch[A])
    H_b = direct_H(cap, B, C, H_pitch[C])
    cap.release()
    w_f = gC / (gA + gC)

    players = traj[(traj.frame == B) & traj["class"].isin(["player", "goalkeeper"])]
    px = players[["x_px", "y_px"]].to_numpy()
    pitch = w_f * map_px(H_f, px) + (1 - w_f) * map_px(H_b, px)

    plt.figure(figsize=(6, 4))
    colors = {"own": "tab:blue", "opp": "tab:red", "unknown": "gray"}
    teams = players["team"].to_numpy()
    for team in set(teams):
        m = teams == team
        plt.scatter(pitch[m, 0], pitch[m, 1], c=colors.get(team, "k"), label=team, s=60)
    plt.axhline(0.333, ls=":", c="gray"); plt.axhline(0.667, ls=":", c="gray")
    plt.xlim(-0.1, 1.1); plt.ylim(-0.1, 1.1); plt.gca().invert_yaxis()
    plt.xlabel("x_pitch"); plt.ylabel("y_pitch (goal-to-goal)")
    plt.title(f"frame {B} blended from anchors {A} & {C} (gaps {gA}/{gC}) — landmark err {eblend:.3f}")
    plt.legend(); plt.tight_layout(); plt.show()